In [0]:
%run ./lab03_config

In [0]:
display(dbutils.fs.ls(staging_root))

In [0]:
display(dbutils.fs.ls(staging_initial_path))

In [0]:
display(dbutils.fs.ls(staging_evolved_path))

In [0]:
initial_df = spark.read.json(staging_initial_path)

display(initial_df)

In [0]:
evolved_df = spark.read.json(staging_evolved_path)

display(evolved_df)

In [0]:
renamed_df = spark.read.json(staging_renamed_path)

display(renamed_df)

In [0]:
malformed_df = spark.read.json(staging_malformed_path)

display(malformed_df)

In [0]:
initial_df.printSchema()

In [0]:
evolved_df.printSchema()

In [0]:
renamed_df.printSchema()


In [0]:
malformed_df.printSchema()

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/dbr_dev/parvinbadalov/lab03_streaming/staging/initial',
  format => 'json'
)
LIMIT 20;

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/dbr_dev/parvinbadalov/lab03_streaming/staging/evolved',
  format => 'json'
)
LIMIT 20;

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/dbr_dev/parvinbadalov/lab03_streaming/staging/renamed',
  format => 'json'
)
LIMIT 20;

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/dbr_dev/parvinbadalov/lab03_streaming/staging/malformed',
  format => 'json'
)
LIMIT 20;

In [0]:
display(dbutils.fs.ls(landing_path))

In [0]:
# Delete the main Bronze table
spark.sql(f"DROP TABLE IF EXISTS {file_bronze_table}")

# Delete Auto Loader progress
dbutils.fs.rm(
    autoloader_checkpoint_path,
    recurse=True
)

# Delete Auto Loader inferred/evolved schema history
dbutils.fs.rm(
    autoloader_schema_path,
    recurse=True
)

# Clear landing
dbutils.fs.rm(
    landing_path,
    recurse=True
)
dbutils.fs.mkdirs(landing_path)

In [0]:
file_count = len(dbutils.fs.ls(landing_path))
display(file_count)

In [0]:
%sql
SELECT *
FROM dbr_dev.parvinbadalov.lab03_taxi_bronze
where
source_system is not null or 
 base_fare_amount is not null
LIMIT 20;

In [0]:
%sql
SELECT COUNT(*) AS total_rows
FROM dbr_dev.parvinbadalov.lab03_taxi_bronze;

In [0]:
%sql
SELECT
    passenger_count,
    _rescued_data,
    _source_file_name
FROM dbr_dev.parvinbadalov.lab03_taxi_rescue_test
WHERE _rescued_data IS NOT NULL
LIMIT 20;

In [0]:
%sql
SELECT
    passenger_count,
    _rescued_data,
    _source_file_name
FROM dbr_dev.parvinbadalov.lab03_taxi_rescue_test
WHERE _rescued_data IS NOT NULL
LIMIT 20;

In [0]:
%sql
SELECT COUNT(*) AS rescued_rows
FROM dbr_dev.parvinbadalov.lab03_taxi_rescue_test
WHERE _rescued_data IS NOT NULL;

In [0]:
%sql
SELECT
    get_json_object(_rescued_data, '$.passenger_count')
        AS rescued_passenger_count,
    get_json_object(_rescued_data, '$.unexpected_field')
        AS rescued_unexpected_field,
    _source_file_name
FROM dbr_dev.parvinbadalov.lab03_taxi_rescue_test
WHERE _rescued_data IS NOT NULL;